In [ ]:
import numpy as np
import pandas as pd

# ==============================================================================
# 1. LOAD PREPROCESSED DATASETS
# ==============================================================================
# Adjust paths if your cleaned CSV files are in '../cleanData/'
loan = pd.read_csv("clean_loan.csv", sep=";")
trans = pd.read_csv("clean_trans.csv", sep=";", low_memory=False)
account = pd.read_csv("clean_account.csv", sep=";")
client = pd.read_csv("clean_client.csv", sep=";")
disp = pd.read_csv("clean_disp.csv", sep=";")
card = pd.read_csv("clean_card.csv", sep=";")
district = pd.read_csv("clean_district.csv", sep=";")
order = pd.read_csv("clean_order.csv", sep=";")

# Ensure IDs are standardized as strings to avoid merge/casting mismatches
id_cols_mapping = {
    loan: ["loan_id", "account_id"],
    trans: ["trans_id", "account_id"],
    account: ["account_id", "district_id"],
    client: ["client_id", "district_id"],
    disp: ["disp_id", "client_id", "account_id"],
    card: ["card_id", "disp_id"],
    order: ["order_id", "account_id"],
}

for df, cols in id_cols_mapping.items():
    for col in cols:
        if col in df.columns:
            df[col] = df[col].astype(str)

district["district_id"] = district["district_id"].astype(str)

# Ensure all date columns are proper pandas datetimes
loan["loan_date"] = pd.to_datetime(loan["date"])
trans["trans_date"] = pd.to_datetime(trans["date"])
account["account_date"] = pd.to_datetime(account["date"])
card["card_issued_date"] = pd.to_datetime(card["issued"])

# Convert Czech client birth_number to datetime if not done already
client["birth_datetime"] = pd.to_datetime(
    client["birth_number"], format="%d-%m-%Y", errors="coerce"
)


# ==============================================================================
# 2. STEP 1: CREATE THE LOAN SPINE
# ==============================================================================
# Spine contains exact unit of analysis (loan_id, account_id), cutoff date, and target
spine = loan[["loan_id", "account_id", "loan_date", "target"]].copy()

# Add loan contract terms directly from loan table
loan_terms = loan[
    ["loan_id", "amount", "duration", "payments", "is_closed"]
].rename(
    columns={"amount": "loan_amount", "payments": "monthly_loan_payment"}
)

print(f"Spine created with {len(spine)} loans.")


# ==============================================================================
# 3. STEP 2: PRE-LOAN TRANSACTION AGGREGATIONS (trans.csv)
# ==============================================================================
# Link transactions to spine to get loan_date boundary
spine_trans = pd.merge(
    spine[["loan_id", "account_id", "loan_date"]],
    trans[
        [
            "account_id",
            "trans_date",
            "type",
            "operation",
            "amount",
            "balance",
            "k_symbol",
        ]
    ],
    on="account_id",
    how="inner",
)

# STRICT FILTER: only events occurring before loan issuance
pre_trans = spine_trans[spine_trans["trans_date"] < spine_trans["loan_date"]].copy()
pre_trans["days_before_loan"] = (
    pre_trans["loan_date"] - pre_trans["trans_date"]
).dt.days


def extract_window_aggregates(df, lookback_days=None, prefix="all"):
    """Computes summary metrics on balance, cash flow, and financial stress."""
    sub = (
        df if lookback_days is None else df[df["days_before_loan"] <= lookback_days]
    ).copy()

    # Cash inflow (credit) vs outflow (withdrawal)
    sub["inflow_amt"] = np.where(sub["type"] == "credit", sub["amount"], 0.0)
    sub["outflow_amt"] = np.where(sub["type"] == "withdrawal", sub["amount"], 0.0)
    sub["is_negative_bal"] = (sub["balance"] < 0).astype(int)

    # Core aggregations per loan
    grouped = sub.groupby("loan_id")
    agg_df = grouped.agg(
        avg_bal=("balance", "mean"),
        min_bal=("balance", "min"),
        max_bal=("balance", "max"),
        std_bal=("balance", "std"),
        tot_inflow=("inflow_amt", "sum"),
        tot_outflow=("outflow_amt", "sum"),
        tx_count=("amount", "count"),
        neg_bal_events=("is_negative_bal", "sum"),
    )

    # Cash flow sustainability ratio
    agg_df["inflow_to_outflow_ratio"] = agg_df["tot_inflow"] / (
        agg_df["tot_outflow"] + 1.0
    )

    # Add lookback window suffix to column names
    agg_df.columns = [f"trans_{col}_{prefix}" for col in agg_df.columns]
    return agg_df


# Calculate over All-Time, 180-Day, and 90-Day windows
trans_all = extract_window_aggregates(pre_trans, lookback_days=None, prefix="all")
trans_180d = extract_window_aggregates(pre_trans, lookback_days=180, prefix="180d")
trans_90d = extract_window_aggregates(pre_trans, lookback_days=90, prefix="90d")

trans_features = (
    trans_all.join(trans_180d, how="left").join(trans_90d, how="left").reset_index()
)


# ==============================================================================
# 4. STEP 3: DEMOGRAPHICS & DISPOSITIONS (disp.csv & client.csv)
# ==============================================================================
# Select only PRIMARY BORROWERS (type == 'OWNER') to avoid duplicate client joins
primary_borrowers = disp[disp["type"] == "OWNER"][
    ["account_id", "client_id", "disp_id"]
]

# Total authorized users (owners + disponents) on account
disp_counts = (
    disp.groupby("account_id")
    .agg(num_authorized_users=("client_id", "count"))
    .reset_index()
)

client_joined = pd.merge(
    spine[["loan_id", "account_id", "loan_date"]],
    primary_borrowers,
    on="account_id",
    how="left",
)
client_joined = pd.merge(
    client_joined,
    client[["client_id", "district_id", "gender", "birth_datetime"]],
    on="client_id",
    how="left",
)

# Compute borrower age at loan issuance
client_joined["borrower_age_at_loan"] = (
    client_joined["loan_date"] - client_joined["birth_datetime"]
).dt.days / 365.25

client_features = client_joined[
    ["loan_id", "district_id", "gender", "borrower_age_at_loan"]
]


# ==============================================================================
# 5. STEP 4: BANK ACCOUNT TENURE & ORDERS (account.csv & order.csv)
# ==============================================================================
# Account tenure at time of loan
account_meta = pd.merge(
    spine[["loan_id", "account_id", "loan_date"]],
    account[["account_id", "frequency", "account_date"]],
    on="account_id",
    how="left",
)
account_meta["account_tenure_days"] = (
    account_meta["loan_date"] - account_meta["account_date"]
).dt.days
account_features = account_meta[
    ["loan_id", "frequency", "account_tenure_days"]
]

# Standing orders / recurring obligations
order_features = (
    order.groupby("account_id")
    .agg(
        num_standing_orders=("order_id", "count"),
        tot_standing_order_amt=("amount", "sum"),
    )
    .reset_index()
)


# ==============================================================================
# 6. STEP 5: CREDIT CARDS ISSUED BEFORE LOAN (card.csv)
# ==============================================================================
# Join cards via owner disp_id and filter strictly before loan date
card_owner = pd.merge(
    spine[["loan_id", "account_id", "loan_date"]],
    primary_borrowers[["account_id", "disp_id"]],
    on="account_id",
    how="left",
)
card_owner = pd.merge(
    card_owner, card[["disp_id", "card_id", "type", "card_issued_date"]], on="disp_id", how="left"
)

pre_loan_cards = card_owner[
    (card_owner["card_id"].notna())
    & (card_owner["card_issued_date"] < card_owner["loan_date"])
]

card_features = (
    pre_loan_cards.groupby("loan_id")
    .agg(has_card_before_loan=("card_id", lambda x: 1 if len(x) > 0 else 0))
    .reset_index()
)


# ==============================================================================
# 7. STEP 6: DISTRICT MACROECONOMIC PROFILE (district.csv)
# ==============================================================================
district_cols = [
    "district_id",
    "population",
    "num_cities",
    "urban_ratio",
    "avg_salary",
    "unemployment_rate_95",
    "unemployment_rate_96",
    "entrepreneurs_per_1000",
    "crimes_95",
    "crimes_96",
]
district_features = district[district_cols].copy()


# ==============================================================================
# 8. STEP 7: ASSEMBLE FINAL FEATURE MATRIX VIA LEFT JOINS
# ==============================================================================
matrix = spine.copy()

# Sequential left joins onto the spine
matrix = matrix.merge(loan_terms, on="loan_id", how="left")
matrix = matrix.merge(account_features, on="loan_id", how="left")
matrix = matrix.merge(disp_counts, on="account_id", how="left")
matrix = matrix.merge(client_features, on="loan_id", how="left")
matrix = matrix.merge(district_features, on="district_id", how="left")
matrix = matrix.merge(order_features, on="account_id", how="left")
matrix = matrix.merge(card_features, on="loan_id", how="left")
matrix = matrix.merge(trans_features, on="loan_id", how="left")

# ==============================================================================
# 9. STEP 8: POST-JOIN IMPUTATION & RISK RATIOS
# ==============================================================================
# Accounts without standing orders or cards default to 0
matrix["has_card_before_loan"] = matrix["has_card_before_loan"].fillna(0).astype(int)
matrix["num_standing_orders"] = matrix["num_standing_orders"].fillna(0).astype(int)
matrix["tot_standing_order_amt"] = matrix["tot_standing_order_amt"].fillna(0.0)

# Financial Capacity Ratios
# Total monthly debt commitments = monthly loan installment + other standing orders
matrix["total_monthly_committed_debt"] = (
    matrix["monthly_loan_payment"] + matrix["tot_standing_order_amt"]
)
matrix["loan_to_tenure_ratio"] = matrix["loan_amount"] / (
    matrix["account_tenure_days"] + 1.0
)
matrix["loan_to_avg_bal_180d"] = matrix["loan_amount"] / (
    matrix["trans_avg_bal_180d"].clip(lower=100.0)
)

# Pipeline verification check
assert len(matrix) == len(
    spine
), f"Row inflation detected! Expected {len(spine)}, got {len(matrix)}"

print(
    f"Successfully built model matrix: {matrix.shape[0]} rows x {matrix.shape[1]} columns."
)

FileNotFoundError: [Errno 2] No such file or directory: 'clean_loan.csv'